# Euchre AI Demonstration Notebook

This notebook demonstrates how the 'dumb' AI players work in the Euchre game.
We'll explore different AI profiles, risk ratios, and decision-making scenarios.

## Setup and Imports

In [ ]:
import sys
import os
from pathlib import Path
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add the parent directory to the path to import euchre modules
sys.path.append(str(Path.cwd().parent))

from euchre.models import Card, Suit, Rank, Player, PlayerType
from euchre.ai.ai_profiles import AggressiveAI, ConservativeAI, BalancedAI, OpportunisticAI
from euchre.ai.base_ai import BaseAI

print("✅ Imports successful")
print(f"🎴 Available suits: {[s.name for s in Suit]}")
print(f"🃏 Available ranks: {[r.name for r in Rank]}")

## AI Profile Overview

The Euchre system has four main AI profiles, each with different playing styles:

In [ ]:
# Create AI players with different profiles
ai_profiles = {
    'Aggressive': AggressiveAI('Aggressive', 0.8),
    'Conservative': ConservativeAI('Conservative', 0.2),
    'Balanced': BalancedAI('Balanced', 0.5),
    'Opportunistic': OpportunisticAI('Opportunistic', 0.6)
}

print("🤖 AI Profiles Created:")
for name, ai in ai_profiles.items():
    print(f"  {name}: risk_ratio={ai.risk_ratio:.1f}")

# Show profile characteristics
profile_info = {
    'Profile': ['Aggressive', 'Conservative', 'Balanced', 'Opportunistic'],
    'Risk Ratio': [0.8, 0.2, 0.5, 0.6],
    'Style': ['High risk, high reward', 'Defensive, safe play', 'Moderate, balanced', 'Adaptive, situational'],
    'Trump Calling': ['Frequent', 'Rare', 'Moderate', 'Context-dependent'],
    'Card Play': ['High cards early', 'Save strong cards', 'Balanced approach', 'Situational']
}

df_profiles = pd.DataFrame(profile_info)
display(df_profiles)

## Scenario 1: Trump Calling Decision

Let's see how different AI profiles decide whether to order up the top card.

In [ ]:
# Create a test scenario for trump calling
def test_trump_calling_scenario():
    """Test trump calling with different AI profiles."""
    
    # Create a test hand
    test_hand = [
        Card(Suit.HEARTS, Rank.ACE),
        Card(Suit.HEARTS, Rank.KING),
        Card(Suit.DIAMONDS, Rank.JACK),  # Left bower for Hearts
        Card(Suit.CLUBS, Rank.QUEEN),
        Card(Suit.SPADES, Rank.TEN)
    ]
    
    # Top card to potentially order up
    top_card = Card(Suit.HEARTS, Rank.JACK)  # Right bower
    
    print("🎴 Test Scenario: Trump Calling Decision")
    print("=" * 50)
    print(f"🃏 Top card: {top_card}")
    print(f"👤 Player hand: {[str(card) for card in test_hand]}")
    print(f"🎯 Potential trump suit: {top_card.suit.name}")
    print(f"🃏 Left bower would be: {ai_profiles['Aggressive']._get_left_bower_suit(top_card.suit).name}")
    print()
    
    # Test each AI profile
    results = []
    for profile_name, ai in ai_profiles.items():
        # Give the AI the test hand
        ai.hand = test_hand.copy()
        
        # Evaluate hand strength
        hand_strength = ai._evaluate_hand_for_trump(top_card.suit, top_card, False)
        
        # Get decision
        should_order = ai.should_order_up(top_card, False)
        
        results.append({
            'Profile': profile_name,
            'Risk Ratio': ai.risk_ratio,
            'Hand Strength': hand_strength,
            'Should Order Up': should_order,
            'Decision': '✅ Order Up' if should_order else '❌ Pass'
        })
        
        print(f"🤖 {profile_name} AI:")
        print(f"  Risk ratio: {ai.risk_ratio:.1f}")
        print(f"  Hand strength: {hand_strength:.1f}")
        print(f"  Decision: {results[-1]['Decision']}")
        print()
    
    # Display results as DataFrame
    df_results = pd.DataFrame(results)
    display(df_results)
    
    return results

# Run the scenario
trump_results = test_trump_calling_scenario()

## Scenario 2: Card Selection Logic

Now let's see how different AI profiles choose which card to play in different situations.

In [ ]:
# Test card selection scenarios
def test_card_selection_scenarios():
    """Test card selection in different game situations."""
    
    print("🎮 Card Selection Scenarios")
    print("=" * 50)
    
    # Scenario 2a: Leading a trick (no lead suit)
    print("\n🎯 Scenario 2a: Leading a Trick (No Lead Suit)")
    print("-" * 40)
    
    test_hand = [
        Card(Suit.HEARTS, Rank.ACE),
        Card(Suit.HEARTS, Rank.KING),
        Card(Suit.DIAMONDS, Rank.JACK),
        Card(Suit.CLUBS, Rank.QUEEN),
        Card(Suit.SPADES, Rank.TEN)
    ]
    
    trump_suit = Suit.HEARTS
    
    print(f"🃏 Hand: {[str(card) for card in test_hand]}")
    print(f"🎯 Trump suit: {trump_suit.name}")
    print(f"🎮 Situation: Leading a trick (no lead suit)")
    print()
    
    for profile_name, ai in ai_profiles.items():
        ai.hand = test_hand.copy()
        
        # Simulate leading (no lead suit)
        chosen_card = ai.choose_card_to_play(None, trump_suit)
        
        print(f"🤖 {profile_name} AI chooses: {chosen_card}")
        
        # Show reasoning
        card_value = ai._card_value(chosen_card, trump_suit)
        print(f"  Card value: {card_value:.1f}")
        
        if chosen_card.is_trump:
            print(f"  Strategy: Playing trump card (high value)")
        elif chosen_card.suit == ai._get_left_bower_suit(trump_suit) and chosen_card.rank == Rank.JACK:
            print(f"  Strategy: Playing left bower (second highest trump)")
        else:
            print(f"  Strategy: Playing high non-trump card")
        print()
    
    # Scenario 2b: Following suit
    print("\n🎯 Scenario 2b: Following Suit")
    print("-" * 40)
    
    lead_suit = Suit.HEARTS
    print(f"🎯 Lead suit: {lead_suit.name}")
    print(f"🎮 Situation: Must follow suit")
    print()
    
    for profile_name, ai in ai_profiles.items():
        ai.hand = test_hand.copy()
        
        # Simulate following suit
        chosen_card = ai.choose_card_to_play(lead_suit, trump_suit)
        
        print(f"🤖 {profile_name} AI chooses: {chosen_card}")
        
        # Show reasoning
        if chosen_card.suit == lead_suit:
            print(f"  Strategy: Following suit with {chosen_card.rank.name}")
        else:
            print(f"  Strategy: Cannot follow suit, playing {chosen_card}")
        print()
    
    # Scenario 2c: Cannot follow suit
    print("\n🎯 Scenario 2c: Cannot Follow Suit")
    print("-" * 40)
    
    # Remove hearts from hand
    no_hearts_hand = [card for card in test_hand if card.suit != Suit.HEARTS]
    
    print(f"🃏 Hand (no hearts): {[str(card) for card in no_hearts_hand]}")
    print(f"🎯 Lead suit: {lead_suit.name}")
    print(f"🎮 Situation: Cannot follow suit, can play any card")
    print()
    
    for profile_name, ai in ai_profiles.items():
        ai.hand = no_hearts_hand.copy()
        
        # Simulate cannot follow suit
        chosen_card = ai.choose_card_to_play(lead_suit, trump_suit)
        
        print(f"🤖 {profile_name} AI chooses: {chosen_card}")
        
        # Show reasoning
        if chosen_card.is_trump:
            print(f"  Strategy: Playing trump to win trick")
        else:
            print(f"  Strategy: Playing high non-trump card")
        print()

# Run the scenarios
test_card_selection_scenarios()

## Scenario 3: Trump Suit Selection

When the top card is rejected, the dealer must choose a trump suit. Let's see how AIs make this decision.

In [ ]:
# Test trump suit selection
def test_trump_suit_selection():
    """Test how AIs choose trump suits when top card is rejected."""
    
    print("🎯 Trump Suit Selection Scenario")
    print("=" * 50)
    
    # Create a test hand with cards in multiple suits
    test_hand = [
        Card(Suit.HEARTS, Rank.ACE),
        Card(Suit.HEARTS, Rank.KING),
        Card(Suit.DIAMONDS, Rank.JACK),
        Card(Suit.DIAMONDS, Rank.QUEEN),
        Card(Suit.CLUBS, Rank.TEN)
    ]
    
    # Top card that was rejected (cannot be chosen)
    rejected_top_card = Card(Suit.SPADES, Rank.KING)
    
    print(f"🃏 Hand: {[str(card) for card in test_hand]}")
    print(f"❌ Rejected top card: {rejected_top_card}")
    print(f"🎮 Situation: Dealer must choose trump suit (cannot be {rejected_top_card.suit.name})")
    print()
    
    # Count cards by suit
    suit_counts = {}
    for suit in Suit:
        if suit != rejected_top_card.suit:  # Exclude rejected suit
            count = len([card for card in test_hand if card.suit == suit])
            suit_counts[suit] = count
    
    print("📊 Available suits and card counts:")
    for suit, count in suit_counts.items():
        print(f"  {suit.name}: {count} cards")
    print()
    
    # Test each AI profile
    for profile_name, ai in ai_profiles.items():
        ai.hand = test_hand.copy()
        
        # Get trump suit choice
        chosen_trump = ai.choose_trump_suit(rejected_top_card)
        
        print(f"🤖 {profile_name} AI chooses: {chosen_trump.name}")
        
        # Show reasoning
        card_count = suit_counts[chosen_trump]
        print(f"  Reasoning: {card_count} cards in {chosen_trump.name}")
        
        # Check if this is the best choice
        best_suit = max(suit_counts.keys(), key=lambda s: suit_counts[s])
        if chosen_trump == best_suit:
            print(f"  ✅ Optimal choice (most cards)")
        else:
            print(f"  ⚠️  Suboptimal choice (best would be {best_suit.name} with {suit_counts[best_suit]} cards)")
        print()

# Run the scenario
test_trump_suit_selection()

## Risk Ratio Analysis

Let's analyze how different risk ratios affect AI decision-making across multiple scenarios.

In [ ]:
# Analyze risk ratio effects
def analyze_risk_ratios():
    """Analyze how different risk ratios affect AI decisions."""
    
    print("🎲 Risk Ratio Analysis")
    print("=" * 50)
    
    # Test different risk ratios for each profile
    risk_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
    
    # Create test scenarios
    test_scenarios = [
        {
            'name': 'Strong Hand',
            'hand': [
                Card(Suit.HEARTS, Rank.ACE),
                Card(Suit.HEARTS, Rank.KING),
                Card(Suit.HEARTS, Rank.QUEEN),
                Card(Suit.DIAMONDS, Rank.JACK),
                Card(Suit.CLUBS, Rank.TEN)
            ],
            'top_card': Card(Suit.HEARTS, Rank.JACK),
            'expected': 'Order Up'
        },
        {
            'name': 'Weak Hand',
            'hand': [
                Card(Suit.HEARTS, Rank.TEN),
                Card(Suit.DIAMONDS, Rank.NINE),
                Card(Suit.CLUBS, Rank.NINE),
                Card(Suit.SPADES, Rank.TEN),
                Card(Suit.HEARTS, Rank.NINE)
            ],
            'top_card': Card(Suit.HEARTS, Rank.JACK),
            'expected': 'Pass'
        },
        {
            'name': 'Marginal Hand',
            'hand': [
                Card(Suit.HEARTS, Rank.ACE),
                Card(Suit.HEARTS, Rank.TEN),
                Card(Suit.DIAMONDS, Rank.NINE),
                Card(Suit.CLUBS, Rank.QUEEN),
                Card(Suit.SPADES, Rank.KING)
            ],
            'top_card': Card(Suit.HEARTS, Rank.JACK),
            'expected': 'Variable'
        }
    ]
    
    # Collect results
    all_results = []
    
    for scenario in test_scenarios:
        print(f"\n🎯 Scenario: {scenario['name']}")
        print(f"🃏 Hand: {[str(card) for card in scenario['hand']]}")
        print(f"🃏 Top card: {scenario['top_card']}")
        print(f"📊 Expected: {scenario['expected']}")
        print("-" * 40)
        
        for profile_name in ['Aggressive', 'Conservative', 'Balanced', 'Opportunistic']:
            for risk_ratio in risk_ratios:
                # Create AI with specific risk ratio
                if profile_name == 'Aggressive':
                    ai = AggressiveAI(f'{profile_name}_{risk_ratio}', risk_ratio)
                elif profile_name == 'Conservative':
                    ai = ConservativeAI(f'{profile_name}_{risk_ratio}', risk_ratio)
                elif profile_name == 'Balanced':
                    ai = BalancedAI(f'{profile_name}_{risk_ratio}', risk_ratio)
                else:  # Opportunistic
                    ai = OpportunisticAI(f'{profile_name}_{risk_ratio}', risk_ratio)
                
                # Give the AI the test hand
                ai.hand = scenario['hand'].copy()
                
                # Get decision
                should_order = ai.should_order_up(scenario['top_card'], False)
                
                # Store result
                all_results.append({
                    'Scenario': scenario['name'],
                    'Profile': profile_name,
                    'Risk Ratio': risk_ratio,
                    'Decision': 'Order Up' if should_order else 'Pass',
                    'Expected': scenario['expected']
                })
                
                print(f"  {profile_name} (risk={risk_ratio:.1f}): {'✅' if should_order else '❌'} {all_results[-1]['Decision']}")
        
        print()
    
    # Create summary DataFrame
    df_risk_analysis = pd.DataFrame(all_results)
    
    print("📊 Risk Ratio Analysis Summary")
    print("=" * 50)
    display(df_risk_analysis)
    
    # Create pivot table
    pivot_table = df_risk_analysis.pivot_table(
        index=['Profile', 'Risk Ratio'],
        columns='Scenario',
        values='Decision',
        aggfunc='first'
    )
    
    print("\n📋 Decision Matrix by Profile and Risk Ratio")
    display(pivot_table)
    
    return df_risk_analysis

# Run the analysis
risk_analysis_results = analyze_risk_ratios()

## Statistical Analysis

Let's run multiple simulations to see how different AI profiles perform statistically.

In [ ]:
# Run statistical analysis
def run_statistical_analysis():
    """Run statistical analysis on AI decision-making."""
    
    print("📊 Statistical Analysis of AI Decision-Making")
    print("=" * 60)
    
    # Generate random hands for testing
    def generate_random_hand() -> List[Card]:
        """Generate a random 5-card hand."""
        all_cards = []
        for suit in Suit:
            for rank in Rank:
                all_cards.append(Card(suit, rank))
        
        return random.sample(all_cards, 5)
    
    def generate_random_top_card() -> Card:
        """Generate a random top card."""
        suit = random.choice(list(Suit))
        rank = random.choice(list(Rank))
        return Card(suit, rank)
    
    # Test parameters
    num_simulations = 100
    risk_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
    
    # Collect data
    simulation_data = []
    
    print(f"🧪 Running {num_simulations} simulations for each profile and risk ratio...")
    
    for profile_name in ['Aggressive', 'Conservative', 'Balanced', 'Opportunistic']:
        for risk_ratio in risk_ratios:
            print(f"\n🤖 Testing {profile_name} with risk ratio {risk_ratio:.1f}...")
            
            # Create AI
            if profile_name == 'Aggressive':
                ai = AggressiveAI(f'{profile_name}_{risk_ratio}', risk_ratio)
            elif profile_name == 'Conservative':
                ai = ConservativeAI(f'{profile_name}_{risk_ratio}', risk_ratio)
            elif profile_name == 'Balanced':
                ai = BalancedAI(f'{profile_name}_{risk_ratio}', risk_ratio)
            else:  # Opportunistic
                ai = OpportunisticAI(f'{profile_name}_{risk_ratio}', risk_ratio)
            
            # Run simulations
            order_up_count = 0
            hand_strengths = []
            
            for sim in range(num_simulations):
                # Generate random scenario
                hand = generate_random_hand()
                top_card = generate_random_top_card()
                
                # Give AI the hand
                ai.hand = hand
                
                # Get decision and hand strength
                should_order = ai.should_order_up(top_card, False)
                hand_strength = ai._evaluate_hand_for_trump(top_card.suit, top_card, False)
                
                if should_order:
                    order_up_count += 1
                
                hand_strengths.append(hand_strength)
                
                # Store data
                simulation_data.append({
                    'Profile': profile_name,
                    'Risk Ratio': risk_ratio,
                    'Simulation': sim,
                    'Hand Strength': hand_strength,
                    'Ordered Up': should_order,
                    'Decision': 'Order Up' if should_order else 'Pass'
                })
            
            # Calculate statistics
            order_up_rate = order_up_count / num_simulations
            avg_hand_strength = np.mean(hand_strengths)
            
            print(f"  📊 Order up rate: {order_up_rate:.1%}")
            print(f"  📊 Average hand strength: {avg_hand_strength:.2f}")
    
    # Create DataFrame
    df_simulations = pd.DataFrame(simulation_data)
    
    print("\n📊 Simulation Results Summary")
    print("=" * 50)
    
    # Summary statistics by profile and risk ratio
    summary_stats = df_simulations.groupby(['Profile', 'Risk Ratio']).agg({
        'Ordered Up': 'mean',
        'Hand Strength': 'mean'
    }).round(3)
    
    summary_stats.columns = ['Order Up Rate', 'Avg Hand Strength']
    display(summary_stats)
    
    # Pivot table for order up rates
    order_up_pivot = df_simulations.pivot_table(
        index='Profile',
        columns='Risk Ratio',
        values='Ordered Up',
        aggfunc='mean'
    ).round(3)
    
    print("\n📋 Order Up Rates by Profile and Risk Ratio")
    display(order_up_pivot)
    
    return df_simulations

# Run the analysis
simulation_results = run_statistical_analysis()

## Visualization

Let's create visualizations to better understand the AI behavior patterns.

In [ ]:
# Create visualizations
def create_visualizations(df_simulations: pd.DataFrame):
    """Create visualizations of AI behavior patterns."""
    
    print("📈 Creating Visualizations")
    print("=" * 50)
    
    # Set up the plotting style
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Euchre AI Behavior Analysis', fontsize=16, fontweight='bold')
    
    # Plot 1: Order Up Rates by Profile and Risk Ratio
    ax1 = axes[0, 0]
    order_up_pivot = df_simulations.pivot_table(
        index='Profile',
        columns='Risk Ratio',
        values='Ordered Up',
        aggfunc='mean'
    )
    
    order_up_pivot.plot(kind='bar', ax=ax1, width=0.8)
    ax1.set_title('Order Up Rates by Profile and Risk Ratio')
    ax1.set_ylabel('Order Up Rate')
    ax1.set_xlabel('AI Profile')
    ax1.legend(title='Risk Ratio', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax1.tick_params(axis='x', rotation=45)
    
    # Plot 2: Hand Strength Distribution
    ax2 = axes[0, 1]
    for profile in df_simulations['Profile'].unique():
        profile_data = df_simulations[df_simulations['Profile'] == profile]
        ax2.hist(profile_data['Hand Strength'], alpha=0.6, label=profile, bins=20)
    
    ax2.set_title('Hand Strength Distribution by Profile')
    ax2.set_xlabel('Hand Strength')
    ax2.set_ylabel('Frequency')
    ax2.legend()
    
    # Plot 3: Risk Ratio vs Order Up Rate
    ax3 = axes[1, 0]
    for profile in df_simulations['Profile'].unique():
        profile_data = df_simulations[df_simulations['Profile'] == profile]
        risk_vs_order = profile_data.groupby('Risk Ratio')['Ordered Up'].mean()
        ax3.plot(risk_vs_order.index, risk_vs_order.values, 'o-', label=profile, linewidth=2, markersize=6)
    
    ax3.set_title('Risk Ratio vs Order Up Rate')
    ax3.set_xlabel('Risk Ratio')
    ax3.set_ylabel('Order Up Rate')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Decision Heatmap
    ax4 = axes[1, 1]
    decision_pivot = df_simulations.pivot_table(
        index='Profile',
        columns='Risk Ratio',
        values='Ordered Up',
        aggfunc='mean'
    )
    
    im = ax4.imshow(decision_pivot.values, cmap='RdYlBu_r', aspect='auto')
    ax4.set_title('Decision Heatmap (Order Up Rate)')
    ax4.set_xlabel('Risk Ratio')
    ax4.set_ylabel('AI Profile')
    ax4.set_xticks(range(len(decision_pivot.columns)))
    ax4.set_xticklabels([f'{r:.1f}' for r in decision_pivot.columns])
    ax4.set_yticks(range(len(decision_pivot.index)))
    ax4.set_yticklabels(decision_pivot.index)
    
    # Add colorbar
    cbar = plt.colorbar(im, ax=ax4)
    cbar.set_label('Order Up Rate')
    
    # Add text annotations to heatmap
    for i in range(len(decision_pivot.index)):
        for j in range(len(decision_pivot.columns)):
            text = ax4.text(j, i, f'{decision_pivot.iloc[i, j]:.2f}',
                           ha='center', va='center', color='black', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Additional analysis: Correlation between hand strength and decisions
    print("\n📊 Correlation Analysis")
    print("-" * 30)
    
    correlation_data = df_simulations.groupby(['Profile', 'Risk Ratio']).agg({
        'Hand Strength': 'corr',
        'Ordered Up': 'corr'
    }).round(3)
    
    print("Correlation between Hand Strength and Order Up decisions:")
    display(correlation_data)

# Create visualizations
if 'simulation_results' in locals():
    create_visualizations(simulation_results)
else:
    print("⚠️  Run the statistical analysis first to generate visualizations.")

## Conclusion

This notebook has demonstrated:

1. **AI Profile Differences**: How Aggressive, Conservative, Balanced, and Opportunistic AIs make decisions
2. **Risk Ratio Effects**: How varying risk ratios (0.1 to 0.9) affect decision-making
3. **Decision Scenarios**: Trump calling, card selection, and trump suit selection
4. **Statistical Analysis**: Performance patterns across multiple simulations
5. **Visual Insights**: Charts showing behavior patterns and correlations

The 'dumb' AIs use rule-based logic with configurable risk parameters to make strategic decisions in Euchre.